# Experiment: Intelligence API THS 巡检

Objective:
- 验证概念板块目录与成分股是否已经走 THS 链路。
- 同时检查原始 THS 页面、AkShare THS 接口、自定义适配层、FastAPI HTTP 接口四层是否一致。

建议：
- 先启动 `python_services`。
- 默认测试 `算力 -> 算力租赁`，也可以改下面配置单元。


In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from io import StringIO
from pathlib import Path
from typing import Any

import akshare as ak
import pandas as pd
import requests
from IPython.display import Markdown, display


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / '.git').exists() and (candidate / 'python_services').exists():
            return candidate
    fallback = Path(os.getenv('STOCK_SCREENING_REPO_ROOT', 'D:/课外项目/stock-screening-boost')).resolve()
    if (fallback / 'python_services').exists():
        return fallback
    raise RuntimeError('找不到仓库根目录，请在仓库根目录启动 Jupyter，或设置 STOCK_SCREENING_REPO_ROOT。')


REPO_ROOT = find_repo_root()
PYTHON_SERVICES_ROOT = REPO_ROOT / 'python_services'
if str(PYTHON_SERVICES_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_SERVICES_ROOT))

from app.services.akshare_adapter import AkShareAdapter

BASE_URL = os.getenv('INTELLIGENCE_BASE_URL', 'http://127.0.0.1:8000').rstrip('/')
REQUEST_TIMEOUT_SECONDS = int(os.getenv('INTELLIGENCE_HTTP_TIMEOUT_SECONDS', '30'))
TEST_THEME = os.getenv('INTELLIGENCE_TEST_THEME', '算力').strip()
TEST_CONCEPT = os.getenv('INTELLIGENCE_TEST_CONCEPT', '算力租赁').strip()
THS_PAGE_INFO_PATTERN = re.compile(r'<span class="page_info">\s*(\d+)\s*/\s*(\d+)\s*</span>')

pd.set_option('display.max_colwidth', 120)


def preview(value: Any, limit: int = 800) -> str:
    if isinstance(value, pd.DataFrame):
        text = value.head(5).to_string(index=False)
    else:
        try:
            text = json.dumps(value, ensure_ascii=False, indent=2, default=str)
        except TypeError:
            text = repr(value)
    return text if len(text) <= limit else f'{text[:limit]} ...'


def http_probe(label: str, method: str, path: str, *, params: dict[str, Any] | None = None, json_body: dict[str, Any] | None = None) -> dict[str, Any]:
    started = time.perf_counter()
    try:
        response = requests.request(method=method, url=f'{BASE_URL}{path}', params=params, json=json_body, timeout=REQUEST_TIMEOUT_SECONDS)
        elapsed_ms = round((time.perf_counter() - started) * 1000, 2)
        try:
            payload = response.json()
        except Exception:
            payload = response.text
        error = payload.get('error', {}) if isinstance(payload, dict) else {}
        meta = payload.get('meta', {}) if isinstance(payload, dict) else {}
        return {
            'label': label,
            'ok': response.ok,
            'status': response.status_code,
            'elapsed_ms': elapsed_ms,
            'provider': meta.get('provider', ''),
            'error_message': error.get('message', '') if isinstance(error, dict) else '',
            'payload': payload,
        }
    except Exception as exc:
        return {
            'label': label,
            'ok': False,
            'status': None,
            'elapsed_ms': round((time.perf_counter() - started) * 1000, 2),
            'provider': '',
            'error_message': str(exc),
            'payload': None,
        }


config = {
    'REPO_ROOT': str(REPO_ROOT),
    'BASE_URL': BASE_URL,
    'TEST_THEME': TEST_THEME,
    'TEST_CONCEPT': TEST_CONCEPT,
    'REQUEST_TIMEOUT_SECONDS': REQUEST_TIMEOUT_SECONDS,
    'akshare_version': getattr(ak, '__version__', 'unknown'),
}
config


In [ ]:
raw_catalog = ak.stock_board_concept_name_ths()
assert not raw_catalog.empty, 'stock_board_concept_name_ths() 返回为空'

selected = raw_catalog[raw_catalog['name'].astype(str).str.strip() == TEST_CONCEPT]
if selected.empty:
    selected = raw_catalog[raw_catalog['name'].astype(str).str.contains(TEST_THEME, regex=False)]
assert not selected.empty, f'在 THS 概念目录中找不到主题/概念: {TEST_THEME} / {TEST_CONCEPT}'

selected_row = selected.iloc[0]
SELECTED_CONCEPT = str(selected_row['name']).strip()
SELECTED_CODE = str(selected_row['code']).strip()

page1_url = f'https://q.10jqka.com.cn/gn/detail/code/{SELECTED_CODE}/'
page1_response = requests.get(page1_url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=20)
page1_response.raise_for_status()
page1_response.encoding = page1_response.apparent_encoding or page1_response.encoding or 'gbk'
page1_html = page1_response.text
page_count_match = THS_PAGE_INFO_PATTERN.search(page1_html)
page_count = int(page_count_match.group(2)) if page_count_match else 1
page1_table = pd.read_html(StringIO(page1_html))[0]

adapter_catalog = AkShareAdapter.get_concept_catalog_frame()
adapter_members = AkShareAdapter.get_concept_constituents_frame(SELECTED_CONCEPT, concept_code=SELECTED_CODE)
assert not adapter_catalog.empty, 'AkShareAdapter.get_concept_catalog_frame() 返回为空'
assert not adapter_members.empty, 'AkShareAdapter.get_concept_constituents_frame() 返回为空'

selected_stock_code = str(adapter_members.iloc[0]['代码']).strip()
raw_summary = pd.DataFrame([
    {'layer': 'raw_ths_catalog', 'ok': True, 'detail': f'catalog_rows={len(raw_catalog)}'},
    {'layer': 'raw_ths_detail_page', 'ok': True, 'detail': f'code={SELECTED_CODE}, page_count={page_count}, page1_rows={len(page1_table)}'},
    {'layer': 'adapter_catalog', 'ok': True, 'detail': f'catalog_rows={len(adapter_catalog)}'},
    {'layer': 'adapter_members', 'ok': True, 'detail': f'concept={SELECTED_CONCEPT}, rows={len(adapter_members)}, first_stock={selected_stock_code}'},
])

display(Markdown(f'**当前选中的 THS 概念** `{SELECTED_CONCEPT}` / code=`{SELECTED_CODE}` / first_stock=`{selected_stock_code}`'))
display(raw_summary)
print('raw_catalog preview:')
print(preview(raw_catalog[['name', 'code']].head(10)))
print('\npage1_table preview:')
print(preview(page1_table))
print('\nadapter_members preview:')
print(preview(adapter_members))


In [ ]:
http_results = [
    http_probe('health', 'GET', '/health'),
    http_probe('v1_concepts', 'GET', f'/api/v1/intelligence/themes/{TEST_THEME}/concepts', params={'limit': 5}),
    http_probe('v1_candidates', 'GET', f'/api/v1/market/themes/{TEST_THEME}/candidates', params={'limit': 5}),
    http_probe('v1_evidence', 'GET', f'/api/v1/intelligence/stocks/{selected_stock_code}/evidence', params={'concept': SELECTED_CONCEPT}),
    http_probe('v1_research_pack', 'GET', f'/api/v1/intelligence/stocks/{selected_stock_code}/research-pack', params={'concept': SELECTED_CONCEPT}),
    http_probe('admin_refresh_concepts', 'POST', '/api/admin/jobs/refresh-concepts', json_body={'limit': 20}),
]

http_df = pd.DataFrame([
    {
        'label': item['label'],
        'ok': item['ok'],
        'status': item['status'],
        'elapsed_ms': item['elapsed_ms'],
        'provider': item['provider'],
        'error_message': item['error_message'],
    }
    for item in http_results
])

display(http_df)
for item in http_results:
    print(f"\n[{item['label']}] preview:")
    print(preview(item['payload']))


In [ ]:
http_by_label = {item['label']: item for item in http_results}

summary = {
    'selected_concept': SELECTED_CONCEPT,
    'selected_code': SELECTED_CODE,
    'selected_stock_code': selected_stock_code,
    'raw_ths_catalog_ok': not raw_catalog.empty,
    'raw_ths_detail_page_ok': not page1_table.empty,
    'adapter_catalog_ok': not adapter_catalog.empty,
    'adapter_members_ok': not adapter_members.empty,
    'http_concepts_ok': http_by_label['v1_concepts']['ok'],
    'http_candidates_ok': http_by_label['v1_candidates']['ok'],
    'http_evidence_ok': http_by_label['v1_evidence']['ok'],
    'http_refresh_concepts_ok': http_by_label['admin_refresh_concepts']['ok'],
}

conclusion_lines = []
if summary['raw_ths_catalog_ok'] and summary['raw_ths_detail_page_ok']:
    conclusion_lines.append('THS 原始目录与详情页本次可用。')
else:
    conclusion_lines.append('THS 原始页层面已经异常，优先看同花顺源站或本机网络。')

if summary['adapter_catalog_ok'] and summary['adapter_members_ok']:
    conclusion_lines.append('自定义 THS 适配层本次可用。')
else:
    conclusion_lines.append('原始 THS 页可用但适配层失败，优先检查 `python_services/app/services/akshare_adapter.py`。')

if summary['http_concepts_ok'] and summary['http_candidates_ok']:
    conclusion_lines.append('FastAPI v1 概念/候选接口本次可用。')
else:
    conclusion_lines.append('适配层可用但 HTTP 失败，优先看 gateway/router/运行时配置。')

display(Markdown('### Notebook summary'))
for line in conclusion_lines:
    print(f'- {line}')
summary


## Next steps

- 如果 `raw_ths_catalog` 就失败：先排查访问 `q.10jqka.com.cn` 的网络连通性。
- 如果原始 THS 页可用但 `adapter_members` 失败：重点检查 [akshare_adapter.py](D:/课外项目/stock-screening-boost/python_services/app/services/akshare_adapter.py)。
- 如果适配层可用但 HTTP 失败：重点看 [intelligence_gateway.py](D:/课外项目/stock-screening-boost/python_services/app/gateway/intelligence_gateway.py) 和 [intelligence_v1.py](D:/课外项目/stock-screening-boost/python_services/app/routers/intelligence_v1.py)。
